# 1. Import libraries and dependencies

In [2]:
from systems import *

# 2. Connect to IBKR Workstation Server

In [3]:
ib_conn = IBKRConnection(live_trading=False)
# ib_conn.connect()
# ib = ib_conn.get_ib()
# print(f"Server time: {ib.reqCurrentTime()}")
ib_conn.disconnect()

Paper trading mode enabled.
Connecting to IBKR at 127.0.0.1:7497 with client ID 1
Disconnecting from IBKR ...


# 3. Backtesting Evaluation

In [4]:
import pandas as pd
import numpy as np
import queue

# 1. Create a dummy DataFrame (simulating what your get_data used to return)
dates = pd.date_range(start='2023-01-01', periods=5, freq='D')
dummy_df = pd.DataFrame({
    'open': [100, 101, 102, 101, 105],
    'high': [102, 103, 104, 105, 108],
    'low': [99, 100, 101, 99, 104],
    'close': [101, 102, 101, 105, 107],
    'volume': [1000, 1500, 1200, 1800, 2000]
}, index=dates)

# 2. Setup the infrastructure
events_queue = queue.Queue()
data_handler = HistoricPandasDataHandler(events_queue, dummy_df, symbol='AAPL')
# data_handler = IBKRLiveDataHandler(events_queue, ib_conn, contract = Stock('AAPL', 'SMART', 'USD'))
print("Starting Backtest Simulation...\n")

# 3. The Main Trading Loop
while data_handler.continue_backtest:
# for i in range(100):
    # Tell the data handler to push the next bar
    data_handler.update_bars()
    
    # Process the events in the queue
    while not events_queue.empty():
        event = events_queue.get()
        if event.type == 'MARKET':
            latest = data_handler.get_latest_bar('AAPL')
            print(f"[{latest['datetime'].date()}] Market Tick - AAPL Close: ${latest['close']:.2f}")
            # Later, your Strategy will live here and look at 'latest' to make decisions.

print("\nBacktest Complete.")

Starting Backtest Simulation...

[2023-01-01] Market Tick - AAPL Close: $101.00
[2023-01-02] Market Tick - AAPL Close: $102.00
[2023-01-03] Market Tick - AAPL Close: $101.00
[2023-01-04] Market Tick - AAPL Close: $105.00
[2023-01-05] Market Tick - AAPL Close: $107.00

Backtest Complete.


In [4]:
import queue
import time
from ib_insync import Stock

# 1. Initialize the Connection (Reusing your exact code!)
print("Starting Live Data Initialization...")
ib_conn = IBKRConnection(live_trading=False) # Use Paper Trading port
ib_conn.connect()

if ib_conn.get_ib().isConnected():
    # 2. Setup the Infrastructure
    events_queue = queue.Queue()
    contract = Stock('AAPL', 'SMART', 'USD')
    
    # Instantiate our new live data handler
    live_data_handler = IBKRLiveDataHandler(events_queue, ib_conn, contract)
    
    # 3. Start the Live Stream
    # This downloads the initial data and opens the live socket
    live_data_handler.start_live_feed()
    
    print("\n--- Listening to Live Market Data ---")
    print("Waiting for the next 1-minute bar to close... (Press Ctrl+C to stop)")
    
    try:
        # 4. The Live Trading Engine Loop
        while True:
            # We must use ib.sleep() instead of time.sleep() in live trading.
            # This allows ib_insync to process the network packets from TWS/Gateway in the background.
            ib_conn.get_ib().sleep(0.1) 
            
            # Check if the callback pushed a new MarketEvent into our queue
            while not events_queue.empty():
                event = events_queue.get()
                
                if event.type == 'MARKET':
                    # Retrieve the actual price data that was saved by the callback
                    latest = live_data_handler.get_latest_bar('AAPL')
                    if latest:
                        timestamp = latest['datetime'].strftime('%H:%M:%S')
                        price = latest['close']
                        print(f"[{timestamp}] TICK FIRED -> AAPL Close: ${price:.2f}")
                        
                        # In the complete system, your Strategy will be triggered right here
                        # strategy.calculate_signals(latest)
                        
    except KeyboardInterrupt:
        print("\nStopping live feed...")
    finally:
        ib_conn.disconnect()
else:
    print("Could not connect to IBKR. Check your TWS/Gateway.")

Starting Live Data Initialization...
Paper trading mode enabled.
Connecting to IBKR at 127.0.0.1:7497 with client ID 1
Connected to IBKR at 127.0.0.1:7497 with client ID 1
Subscribing to live data for AAPL...

--- Listening to Live Market Data ---
Waiting for the next 1-minute bar to close... (Press Ctrl+C to stop)

Stopping live feed...
Disconnecting from IBKR ...


# 4. Testing strategies

In [5]:
import pandas as pd
import queue
# from events import SignalEvent
from systems import HistoricPandasDataHandler
from strategy import *
# 1. Create a dummy price dataset with a clear crossover
# Prices start low, trend high (creating a Buy), then crash (creating a Sell)
dates = pd.date_range(start='2023-01-01', periods=8, freq='D')
dummy_df = pd.DataFrame({
    'open': [10, 11, 12, 13, 14, 15, 9, 8],
    'high': [10, 11, 12, 13, 14, 15, 9, 8],
    'low': [10, 11, 12, 13, 14, 15, 9, 8],
    'close': [10, 11, 12, 13, 14, 15, 9, 8], # Focus on the close
}, index=dates)

# 2. Setup the infrastructure
events_queue = queue.Queue()

# Initialize the DataHandler
data_handler = HistoricPandasDataHandler(events_queue, dummy_df, symbol='AAPL')

# Initialize the Strategy (Using short periods so we can see it trigger quickly)
# Fast MA = 2 periods, Slow MA = 4 periods
mac_strategy = MovingAverageStrategy(events_queue, data_handler, symbol='AAPL', fast_period=2, slow_period=4)

print("Starting Strategy Engine Loop...\n")

# 3. The Main Trading Loop
while data_handler.continue_backtest:
    # Tell the data handler to push the next bar (This puts a MarketEvent in the queue)
    data_handler.update_bars()
    
    # Process all events in the queue
    while not events_queue.empty():
        event = events_queue.get()
        
        if event.type == 'MARKET':
            latest = data_handler.get_latest_bar('AAPL')
            print(f"Market Tick -> Date: {latest['datetime'].date()} | Price: ${latest['close']:.2f}")
            
            # The Engine passes the MarketEvent to the strategy
            mac_strategy.calculate_signals(event)
            
        elif event.type == 'SIGNAL':
            print(f">>> CAUGHT IN QUEUE: {event.signal_type} Signal for {event.symbol} from {event.strategy_id} <<<")
            # In the next step, the Portfolio Manager will pick this up!

print("\nSimulation Complete.")

Starting Strategy Engine Loop...

Market Tick -> Date: 2023-01-01 | Price: $10.00
Market Tick -> Date: 2023-01-02 | Price: $11.00
Market Tick -> Date: 2023-01-03 | Price: $12.00
Market Tick -> Date: 2023-01-04 | Price: $13.00
[2023-01-04 00:00:00] SIGNAL: Fast MA (12.50) > Slow MA (11.50). Going LONG.
>>> CAUGHT IN QUEUE: LONG Signal for AAPL from MA_Cross_1 <<<
Market Tick -> Date: 2023-01-05 | Price: $14.00
Market Tick -> Date: 2023-01-06 | Price: $15.00
Market Tick -> Date: 2023-01-07 | Price: $9.00
[2023-01-07 00:00:00] SIGNAL: Fast MA (12.00) < Slow MA (12.75). Going SHORT/FLAT.
>>> CAUGHT IN QUEUE: SHORT Signal for AAPL from MA_Cross_1 <<<
Market Tick -> Date: 2023-01-08 | Price: $8.00

Simulation Complete.


# 5. Portfolio Manager

In [6]:
from portfolio_manager import PortfolioManager

import pandas as pd
import queue
# (Assuming Event, MarketEvent, SignalEvent, OrderEvent, HistoricPandasDataHandler, and MovingAverageStrategy are loaded)

# 1. Setup Dummy Data (Clear MA crossover pattern)
dates = pd.date_range(start='2023-01-01', periods=8, freq='D')
dummy_df = pd.DataFrame({
    'open': [10, 11, 12, 13, 14, 15, 9, 8],
    'high': [10, 11, 12, 13, 14, 15, 9, 8],
    'low':  [10, 11, 12, 13, 14, 15, 9, 8],
    'close':[10, 11, 12, 13, 14, 15, 9, 8], 
}, index=dates)

# 2. Initialize the Infrastructure
events_queue = queue.Queue()

# Instantiate the layers
data_handler = HistoricPandasDataHandler(events_queue, dummy_df, symbol='AAPL')
mac_strategy = MovingAverageStrategy(events_queue, data_handler, symbol='AAPL', fast_period=2, slow_period=4)
portfolio = PortfolioManager(events_queue, data_handler, initial_capital=10000.0)

print("Starting Trading Engine...\n")

# 3. The Main Event Loop
while data_handler.continue_backtest:
    # Trigger the next bar
    data_handler.update_bars()
    
    # Process the queue until it is completely empty
    while not events_queue.empty():
        event = events_queue.get()
        
        if event.type == 'MARKET':
            # 1. Market data arrives, pass it to the strategy
            latest = data_handler.get_latest_bar('AAPL')
            print(f"\nTick: Date {latest['datetime'].date()} | Price: ${latest['close']:.2f}")
            mac_strategy.calculate_signals(event)
            
        elif event.type == 'SIGNAL':
            # 2. Strategy generated a signal, pass it to the portfolio
            portfolio.update_signal(event)
            
        elif event.type == 'ORDER':
            # 3. Portfolio approved it and sized it. Next stop: The Broker!
            print(f">>> CAUGHT IN QUEUE: Final Order -> {event.direction} {event.quantity} {event.symbol} @ {event.order_type} <<<")
            
print("\nSimulation Complete.")

Starting Trading Engine...


Tick: Date 2023-01-01 | Price: $10.00

Tick: Date 2023-01-02 | Price: $11.00

Tick: Date 2023-01-03 | Price: $12.00

Tick: Date 2023-01-04 | Price: $13.00
[2023-01-04 00:00:00] SIGNAL: Fast MA (12.50) > Slow MA (11.50). Going LONG.
[PORTFOLIO] Approved LONG signal. Generated BUY order for 10 AAPL.
>>> CAUGHT IN QUEUE: Final Order -> BUY 10 AAPL @ MKT <<<

Tick: Date 2023-01-05 | Price: $14.00

Tick: Date 2023-01-06 | Price: $15.00

Tick: Date 2023-01-07 | Price: $9.00
[2023-01-07 00:00:00] SIGNAL: Fast MA (12.00) < Slow MA (12.75). Going SHORT/FLAT.

Tick: Date 2023-01-08 | Price: $8.00

Simulation Complete.


# 6. Execution

In [ ]:
import pandas as pd
import queue

from execution import SimulatedExecutionHandler

# 1. Setup Dummy Data (Clear MA crossover pattern)
dates = pd.date_range(start='2023-01-01', periods=8, freq='D')
dummy_df = pd.DataFrame({
    'open': [10, 11, 12, 13, 14, 15, 9, 8],
    'high': [10, 11, 12, 13, 14, 15, 9, 8],
    'low':  [10, 11, 12, 13, 14, 15, 9, 8],
    'close':[10, 11, 12, 13, 14, 15, 9, 8], 
}, index=dates)

# 2. Initialize the Infrastructure
events_queue = queue.Queue()

data_handler = HistoricPandasDataHandler(events_queue, dummy_df, symbol='AAPL')
mac_strategy = MovingAverageStrategy(events_queue, data_handler, symbol='AAPL', fast_period=2, slow_period=4)
portfolio = PortfolioManager(events_queue, data_handler, initial_capital=10000.0)
execution = SimulatedExecutionHandler(events_queue, data_handler)

print("Starting Full Trading Engine Loop...\n")

# 3. The Complete Event Loop
while data_handler.continue_backtest:
    data_handler.update_bars()
    
    while not events_queue.empty():
        event = events_queue.get()
        
        if event.type == 'MARKET':
            mac_strategy.calculate_signals(event)
            
        elif event.type == 'SIGNAL':
            portfolio.update_signal(event)
            
        elif event.type == 'ORDER':
            execution.execute_order(event)
            
        elif event.type == 'FILL':
            # This is the final piece! The portfolio updates its cash balance
            # using the actual fill price from the broker.
            portfolio.update_fill(event)

print("\nSimulation Complete.")
print(f"Final Portfolio Cash: ${portfolio.current_cash:.2f}")
print(f"Final Holdings: {portfolio.holdings}")

Starting Full Trading Engine Loop...

[2023-01-04 00:00:00] SIGNAL: Fast MA (12.50) > Slow MA (11.50). Going LONG.
[PORTFOLIO] Approved LONG signal. Generated BUY order for 10 AAPL.
[EXECUTION - SIM] FILLED BUY 10 AAPL @ $13.00
[PORTFOLIO] Fill received. New Cash Balance: $9869.00 | Holdings: {'AAPL': 10}
[2023-01-07 00:00:00] SIGNAL: Fast MA (12.00) < Slow MA (12.75). Going SHORT/FLAT.
[PORTFOLIO] Approved SHORT/EXIT signal. Generated SELL order to close 10 AAPL.
[EXECUTION - SIM] FILLED SELL 10 AAPL @ $9.00
[PORTFOLIO] Fill received. New Cash Balance: $9958.00 | Holdings: {'AAPL': 0}

Simulation Complete.
Final Portfolio Cash: $9958.00
Final Holdings: {'AAPL': 0}


In [9]:
import queue
import time
from datetime import datetime
from ib_insync import Stock, MarketOrder
from execution import IBKRExecutionHandler 

# --- ASSUMING THESE CLASSES ARE DEFINED FROM PREVIOUS STEPS ---
# Event, OrderEvent, FillEvent, PortfolioManager, ExecutionHandler, IBKRExecutionHandler
# And your original IBKRConnection class.

print("--- Starting Live IBKR Execution Test ---")

# 1. Initialize the Connection
# Using your exact connection class from the beginning!
ib_conn = IBKRConnection(live_trading=False) # live_trading=False uses the paper port
ib_conn.connect()

if ib_conn.get_ib().isConnected():
    try:
        # 2. Setup the Infrastructure
        events_queue = queue.Queue()
        
        # We need a dummy data handler just so the PortfolioManager doesn't crash 
        # (in a real system, this would be your IBKRLiveDataHandler)
        class DummyDataHandler:
            def get_latest_bar(self, symbol): return {'close': 150.0} # Fake price
            
        data_handler = DummyDataHandler()
        
        # Initialize Portfolio and Execution engines
        portfolio = PortfolioManager(events_queue, data_handler, initial_capital=100000.0)
        execution = IBKRExecutionHandler(events_queue, ib_conn)
        
        # 3. Inject a Manual Order (Simulating the Strategy/Portfolio output)
        print("\n[SYSTEM] Injecting manual BUY order for 10 AAPL into the queue...")
        manual_order = OrderEvent(symbol='AAPL', order_type='MKT', quantity=10, direction='BUY')
        events_queue.put(manual_order)
        
        # 4. The Event Loop
        print("\n[SYSTEM] Starting Event Loop to process queue...")
        
        while not events_queue.empty():
            event = events_queue.get()
            
            if event.type == 'ORDER':
                print(f"\n---> Loop caught ORDER Event for {event.quantity} {event.symbol}. Routing to Execution Handler...")
                # The Execution Handler talks to IBKR, places the trade, waits for the fill, 
                # and drops a FILL event back into this exact queue.
                execution.execute_order(event)
                
            elif event.type == 'FILL':
                print(f"\n---> Loop caught FILL Event. Routing to Portfolio Manager...")
                # The Portfolio Manager updates your cash balance and holdings
                portfolio.update_fill(event)
                
                # Print the final result
                print("\n--- Final Portfolio State ---")
                print(f"Cash Remaining: ${portfolio.current_cash:.2f}")
                print(f"Current Holdings: {portfolio.holdings}")
                
    except Exception as e:
        print(f"An error occurred: {e}")
    finally:
        print("\nDisconnecting from IBKR...")
        ib_conn.disconnect()
else:
    print("Failed to connect to IBKR. Please check TWS/Gateway.")

--- Starting Live IBKR Execution Test ---
Paper trading mode enabled.
Connecting to IBKR at 127.0.0.1:7497 with client ID 1
Connected to IBKR at 127.0.0.1:7497 with client ID 1

[SYSTEM] Injecting manual BUY order for 10 AAPL into the queue...

[SYSTEM] Starting Event Loop to process queue...

---> Loop caught ORDER Event for 10 AAPL. Routing to Execution Handler...
[EXECUTION - IBKR] Sending BUY order for 10 AAPL to broker...


Error 10349, reqId 4: Order TIF was set to DAY based on order preset.
Canceled order: Trade(contract=Stock(conId=265598, symbol='AAPL', exchange='SMART', primaryExchange='NASDAQ', currency='USD', localSymbol='AAPL', tradingClass='NMS'), order=MarketOrder(orderId=4, clientId=1, action='BUY', totalQuantity=10), orderStatus=OrderStatus(orderId=4, status='Cancelled', filled=0.0, remaining=0.0, avgFillPrice=0.0, permId=0, parentId=0, lastFillPrice=0.0, clientId=0, whyHeld='', mktCapPrice=0.0), fills=[], log=[TradeLogEntry(time=datetime.datetime(2026, 3, 5, 11, 30, 25, 446314, tzinfo=datetime.timezone.utc), status='PendingSubmit', message='', errorCode=0), TradeLogEntry(time=datetime.datetime(2026, 3, 5, 11, 30, 42, 822427, tzinfo=datetime.timezone.utc), status='Cancelled', message='Error 10349, reqId 4: Order TIF was set to DAY based on order preset.', errorCode=10349)], advancedError='')


[EXECUTION - IBKR] Order failed or cancelled. Status: Cancelled

Disconnecting from IBKR...
Disconnecting from IBKR ...


# 7. Evaluation

In [14]:
# ==========================================
# Example Usage Integration
# ==========================================

# Mocking the DataFrame generation from your original MovingAverageStrategy
from evaluation import QuantitativeEvaluator

np.random.seed(42)
dates = pd.date_range(start='2023-01-01', periods=100, freq='D')

# Simulating a strategy that generated a cumulative return column
# In reality, this comes directly from: df = strat.test_strategy()
mock_cumulative_returns = np.cumprod(1 + np.random.normal(0.001, 0.015, 100))

df = pd.DataFrame({
    'close': np.random.uniform(100, 150, 100),
    'Strategy_Return': mock_cumulative_returns
}, index=dates)

# Initialize the separated Evaluator class
evaluator = QuantitativeEvaluator(risk_free_rate=0.02)

# Calculate metrics
performance_report = evaluator.calculate_metrics(df)

print("--- Strategy Performance Report ---")
for metric, value in performance_report.items():
    print(f"{metric}: {value}")
    
# Plot the drawdown
evaluator.plot_drawdown(df)

--- Strategy Performance Report ---
Total Return: -6.29%
Annualized Volatility: 21.69%
Sharpe Ratio: -0.846
Sortino Ratio: -1.306
Max Drawdown: -18.83%
Win Rate (Days): 47.00%


TypeError: 'module' object is not callable